In [11]:

from datetime import datetime
from pyspark.sql.types import *
import uuid

# === CONFIGURATION - Change for each notebook ===
NOTEBOOK_NAME = "ntk_Sil2Gld_DimPermission V2"        # ← Change this for each notebook
PIPELINE_NAME = "pipeline_test"     # ← Change this for each pipeline
ACTIVITY_TYPE = "DataTransformation"     # ← DataExtract/DataTransform/DataLoad/DataValidation
SOURCE_PATH = "abfss://Silver/Dim_Permission" # ← Change source path
TARGET_PATH = "abfss://Gold/Dim_Permission" # ← Change target path

def log_etl_activity(status, start_time=None, error=None, **metrics):
    """Log ETL activity to pipeline table"""
    current_time = datetime.now()
    
    # Get next LogID
    try:
        log_id = spark.sql("SELECT COALESCE(MAX(LogID), 0) + 1 as id FROM etl_gold_pipeline_log").collect()[0]['id']
    except:
        log_id = 1
    
    if status == "STARTED":
        data = [(
            log_id, PIPELINE_NAME, f"run_{current_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            current_time, None, None, "RUNNING", None, 
            SOURCE_PATH, TARGET_PATH, None, None, None, None, 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
        start_time = current_time
        
    else:
        duration = int((current_time - start_time).total_seconds()) if start_time else None
        data = [(
            log_id, PIPELINE_NAME, f"run_{start_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            start_time, current_time, duration, status, 
            str(error) if error else None, SOURCE_PATH, TARGET_PATH,
            metrics.get('rows_read'), metrics.get('rows_written'), 
            metrics.get('file_count'), metrics.get('bytes_processed'), 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
    
    # Schema for etl_gold_pipeline_log table
    schema = StructType([
        StructField("LogID", LongType()), StructField("PipelineName", StringType()),
        StructField("RunID", StringType()), StructField("ActivityName", StringType()),
        StructField("ActivityType", StringType()), StructField("NotebookName", StringType()),
        StructField("Sequence", IntegerType()), StructField("StartTime", TimestampType()),
        StructField("EndTime", TimestampType()), StructField("DurationSeconds", IntegerType()),
        StructField("Status", StringType()), StructField("ErrorMessage", StringType()),
        StructField("SourcePath", StringType()), StructField("TargetPath", StringType()),
        StructField("RowsRead", LongType()), StructField("RowsWritten", LongType()),
        StructField("FileCountProcessed", IntegerType()), StructField("BytesProcessed", LongType()),
        StructField("InsertedOn", TimestampType()), StructField("InsertedBy", StringType()),
        StructField("CorrelationID", StringType())
    ])
    
    # Save to table
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable("etl_gold_pipeline_log")
    
    # Print status
    if status == "STARTED":
        print(f"🚀 Starting {NOTEBOOK_NAME}")
    elif status == "SUCCESS":
        duration_text = f" ({duration}s)" if duration else ""
        print(f"✅ {NOTEBOOK_NAME} completed successfully{duration_text}")
    else:
        print(f"❌ {NOTEBOOK_NAME} failed")
    
    return current_time if status == "STARTED" else None

# Start logging
print(f"🔧 Initializing {NOTEBOOK_NAME}...")
start_time = log_etl_activity("STARTED")

# Initialize variables for tracking metrics
rows_read = 0
rows_written = 0
file_count = 0
bytes_processed = 0

# print(f"📊 Ready to process data from: {SOURCE_PATH}")
# print(f"🎯 Target location: {TARGET_PATH}")

StatementMeta(, 0f183f49-359e-4c2e-89a4-eb28568aa39f, 13, Finished, Available, Finished)

🔧 Initializing ntk_Sil2Gld_DimPermission V2...
🚀 Starting ntk_Sil2Gld_DimPermission V2


In [12]:
# Section --- Importing modules and Defining Variable for standard usage source / target.
from pyspark.sql import SparkSession
from datetime import datetime
import os

# Base source path (up to Files level)
base_source_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files"
# Base target path  
base_target_path  = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files"

# Get today's date and format it
today = datetime.now()
year = today.strftime("%Y")
month = today.strftime("%m")
day = today.strftime("%d")

# Build dynamic source folder structure
sourcefolder_structure = f"Silver_layer/Reporting/{year}/{month}/{day}"   
# Complete source path
complete_source_path = f"{base_source_path}/{sourcefolder_structure}"
# Filename
source_filename = "Dim_Permission.parquet"

# Build dynamic target folder structure
targetfolder_structure = f"/PreGold_Reporting"   
# Complete target path
complete_target_path = f"{base_target_path}/{targetfolder_structure}"
# Filename
target_filename = "Dim_Permission.parquet"
print(f"Variables created and session started.")



StatementMeta(, 0f183f49-359e-4c2e-89a4-eb28568aa39f, 14, Finished, Available, Finished)

Variables created and session started.


In [13]:
# Initialize Spark session
spark = SparkSession.builder.appName("SilverToGold_FileReader").getOrCreate()

# Full Source  file path
full_source_path = f"{complete_source_path}/{source_filename}"

# Full Target  file path
full_target_path = f"{complete_target_path}/{target_filename}"

print(f"Source: {full_source_path}")
print(f"Target: {full_target_path}")

StatementMeta(, 0f183f49-359e-4c2e-89a4-eb28568aa39f, 15, Finished, Available, Finished)

Source: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting/2025/10/06/Dim_Permission.parquet
Target: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_Permission.parquet


In [14]:
# mssparkutils.notebook.run("nbk_dimuser_validations", 60)

StatementMeta(, 0f183f49-359e-4c2e-89a4-eb28568aa39f, 16, Finished, Available, Finished)

In [15]:

# ===== READ PROCESS =====
try:
    # Read the Parquet file from Silver layer
    print("Reading from Silver layer...")
    df_silver = spark.read.parquet(full_source_path)
    print(f"Input file read with total records: {df_silver.count()}")
    
    df_silver.show(1)

    # STAGE 1: PRE-VALIDATION GATE - CRITICAL CHECKPOINT
    # validation_passed, validation_report = step1_validate_source_quality(df_silver)

    # STOP PIPELINE IF CRITICAL VALIDATION FAILURES
    # if not validation_passed:
    #     print("\n🛑 ETL PIPELINE STOPPED - Critical validation failures detected")
    #     print("Fix source data issues before proceeding")
    #     raise Exception("Pre-validation failed - ETL pipeline terminated")
        
    # print("\n🚀 Pre-validation passed - Proceeding with transformation...")

except Exception as e:
    print(f"Error in processing: {str(e)}")
    print("Please check:")
    print(f"1. Source file exists: {full_source_path}")
    print(f"2. Target path is accessible: {base_target_path}")

print("Process completed!")


StatementMeta(, 0f183f49-359e-4c2e-89a4-eb28568aa39f, 17, Finished, Available, Finished)

Reading from Silver layer...
Input file read with total records: 10
+------------+-----------------+----------+-----+-------------+--------+-----------+---------------+------------+-------------+-------------+---------+------------+------------+---------+-----------------+--------+---------------+--------------------+--------------------+----------+--------------------+
|        Name|      Description|        Id|Order|         Type|IsHidden|ManageLists|DeleteListItems|AddListItems|EditListItems|ViewListItems|OpenItems|ViewVersions|CreateAlerts|ViewPages|ManagePermissions|FullMask|   CapturedDate|        SnapshotDate|       ProcessedDate|DataSource|       PermissionKey|
+------------+-----------------+----------+-----+-------------+--------+-----------+---------------+------------+-------------+-------------+---------+------------+------------+---------+-----------------+--------+---------------+--------------------+--------------------+----------+--------------------+
|Full Control|Has

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

# ===== ETL TRANSFORMATION PLAN =====

def transform_silver_to_gold(df_silver):
    """
    Transform Silver layer Dim_User to Gold layer reporting format
    """
    
    # 1. COLUMN SELECTION & DIRECT MAPPING
    df_transformed = df_silver.select(
        col("PermissionKey"),
        col("Id").alias("PermissionID"),
        col("Name").alias("PermissionName"),
        col("Type").alias("PermissionType"),
        col("ProcessedDate"),
        col("DataSource"),
        col("SnapshotDate"),
        # col("OwnerLoginName").alias("Owner"),
        col("Description").alias("PermissionDesc")
    )
    
    # 2. ADD MISSING COLUMNS WITH DEFAULT VALUES
    df_gold = df_transformed \
        .withColumn("PrincipalType", lit(None).cast(StringType())) \
        .withColumn("GrantedVia", lit(None).cast(StringType())) \
        .withColumn("PermissionScope", lit(None).cast(StringType())) \
        .withColumn("PermissionStatus", lit(None).cast(StringType())) \
        .withColumn("GrantedVia", lit(None).cast(StringType())) \
        .withColumn("Ownerid", lit(None).cast(StringType())) \
        .withColumn("OwnerEmail", lit(None).cast(StringType())) \
        .withColumn("IsExternal", lit(None).cast(StringType())) \
        .withColumn("CreatedBy", lit("System").cast(StringType())) \
        .withColumn("CreatedDate", current_date()) \
        .withColumn("LastModifiedBy", lit("System").cast(StringType())) \
        .withColumn("ModifiedDate", current_date())

    # 3. REORDER COLUMNS TO MATCH OUTPUT STRUCTURE
    df_final = df_gold.select(
        "PermissionKey",
        "PermissionID",
        "PermissionName",
        "PermissionType",
        "PermissionScope",
        "PermissionDesc",
        "PermissionStatus",
        "GrantedVia",
        "PrincipalType",
        "Ownerid",
        "OwnerEmail",
        "IsExternal",
        "ProcessedDate",
        "DataSource",
        "CreatedBy",
        "CreatedDate",
        "LastModifiedBy",
        "ModifiedDate",
        "SnapshotDate"
    )
    
    return df_final

# Apply transformation
df_gold_ready = transform_silver_to_gold(df_silver)


# # ===== TRANSFORM PROCESS =====
# # Step 1: Split permissionDesc column into array
# df_transformed = (
#     df_gold_ready
#     .withColumn("permissionArray", F.split(F.col("permissionDesc"), ",\\s*"))
# )

# # Step 2: Explode array into multiple rows
# df_exploded = df_transformed.withColumn("permissionDescExploded", F.explode(F.col("permissionArray")))

# # Step 3: Trim spaces (important for clean values)
# df_exploded = df_exploded.withColumn("permissionDescExploded", F.trim(F.col("permissionDescExploded")))

# # Step 4: Filter out blanks or nulls
# df_gold_ready_final = df_exploded.filter(
#     (F.col("permissionDescExploded").isNotNull()) &
#     (F.col("permissionDescExploded") != "")
# ).drop("permissionArray")

# # Drop intermediate array column if not needed
# df_gold_ready_final = df_exploded.drop("permissionArray")

# Show results
print("Gold layer transformation completed!")
df_gold_ready.show(5, truncate=False)
df_gold_ready.printSchema()

StatementMeta(, 0f183f49-359e-4c2e-89a4-eb28568aa39f, 9, Finished, Available, Finished)

Gold layer transformation completed!
+----------------------------------------------------------------+------------+--------------+--------------+---------------+------------------------------------------------------------------------------------------+----------------+----------+-------------+-------+----------+----------+--------------------------+----------+---------+-----------+--------------+------------+--------------------------+
|PermissionKey                                                   |PermissionID|PermissionName|PermissionType|PermissionScope|PermissionDesc                                                                            |PermissionStatus|GrantedVia|PrincipalType|Ownerid|OwnerEmail|IsExternal|ProcessedDate             |DataSource|CreatedBy|CreatedDate|LastModifiedBy|ModifiedDate|SnapshotDate              |
+----------------------------------------------------------------+------------+--------------+--------------+---------------+------------------------------

In [8]:
# ===== WRITE PROCESS =====
try:
    # Write to Gold layer (PreGold_Reporting)
    print(f"Writing to Gold layer: {full_target_path}")
    
    df_gold_ready.write \
        .mode("overwrite") \
        .option("compression", "snappy") \
        .parquet(full_target_path)
    
    print(f"Successfully written to: {full_target_path}")
    
    # Verify the written file
    df_verify = spark.read.parquet(full_target_path)
    print(f"Verification - Target record count: {df_verify.count()}")
    
except Exception as e:
    print(f"Error in processing: {str(e)}")
    print("Please check:")
    print(f"1. Source file exists: {full_source_path}")
    print(f"2. Target path is accessible: {base_target_path}")

print("Process completed!")


StatementMeta(, 0f183f49-359e-4c2e-89a4-eb28568aa39f, 10, Finished, Available, Finished)

Writing to Gold layer: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_Permission.parquet
Successfully written to: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_Permission.parquet
Verification - Target record count: 10
Process completed!


In [8]:
df_verify.printSchema()

StatementMeta(, 6d5957a4-bec8-41d0-a260-86af40f79678, 10, Finished, Available, Finished)

root
 |-- PermissionKey: string (nullable = true)
 |-- PermissionID: string (nullable = true)
 |-- PermissionName: string (nullable = true)
 |-- PermissionType: string (nullable = true)
 |-- PermissionScope: string (nullable = true)
 |-- PermissionDesc: string (nullable = true)
 |-- PermissionStatus: string (nullable = true)
 |-- GrantedVia: string (nullable = true)
 |-- PrincipalType: string (nullable = true)
 |-- Ownerid: string (nullable = true)
 |-- OwnerEmail: string (nullable = true)
 |-- IsExternal: string (nullable = true)
 |-- ProcessedDate: timestamp (nullable = true)
 |-- DataSource: string (nullable = true)
 |-- CreatedBy: string (nullable = true)
 |-- CreatedDate: date (nullable = true)
 |-- LastModifiedBy: string (nullable = true)
 |-- ModifiedDate: date (nullable = true)
 |-- SnapshotDate: timestamp (nullable = true)



In [16]:

# Ensure processing_successful is defined before this block
try:
    print(f"🔄 Starting ETL processing for {NOTEBOOK_NAME}...")

    # Your ETL logic here
    # Example:
    # source_df = spark.read.format("delta").load(SOURCE_PATH)
    # transformed_df = source_df.filter("status = 'active'")
    # transformed_df.write.format("delta").mode("overwrite").save(TARGET_PATH)

    # Simulated metrics
    rows_read = df_silver.count()
    rows_written = df_gold_ready.count()
    # file_count = len(dbutils.fs.ls(SOURCE_PATH))
    bytes_processed = 524288000  # ~500MB

    processing_successful = True

except Exception as e:
    error_details = e
    processing_successful = False

# Complete the logging based on processing results
if processing_successful:
    # Log successful completion with metrics
    log_etl_activity("SUCCESS", start_time, 
                     rows_read=rows_read, 
                     rows_written=rows_written,
                    #  file_count=file_count,
                     bytes_processed=bytes_processed)
    
    print(f"🎉 {NOTEBOOK_NAME} pipeline completed successfully!")
    print(f"📊 Final metrics:")
    print(f"   ✅ Status: SUCCESS")
    print(f"   📖 Total rows processed: {rows_read:,} → {rows_written:,}")
    print(f"   🔄 Data throughput: {bytes_processed/(1024**2):.1f} MB")
    
    # Optional: Show recent logs for this notebook
    print(f"\n📋 Recent runs for {NOTEBOOK_NAME}:")
    spark.sql(f"""
        SELECT LogID, Status, StartTime, EndTime, DurationSeconds, RowsRead, RowsWritten
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=True)
    
else:
    # Log failure
    log_etl_activity("FAILED", start_time, error_details)
    
    print(f"💥 {NOTEBOOK_NAME} pipeline failed!")
    print(f"❌ Error: {str(error_details)}")
    
    # Optional: Show error analysis
    print(f"\n🔍 Recent failures for debugging:")
    spark.sql(f"""
        SELECT LogID, StartTime, ErrorMessage, DurationSeconds
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}' AND Status = 'FAILED'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=False)
    
    # Re-raise the exception to fail the notebook
    raise error_details

# Cleanup variables
print(f"\n🧹 Cleaning up variables...")
del rows_read, rows_written, bytes_processed

print(f"✨ {NOTEBOOK_NAME} logging completed!")


StatementMeta(, 0f183f49-359e-4c2e-89a4-eb28568aa39f, 18, Finished, Available, Finished)

🔄 Starting ETL processing for ntk_Sil2Gld_DimPermission V2...
✅ ntk_Sil2Gld_DimPermission V2 completed successfully (49s)
🎉 ntk_Sil2Gld_DimPermission V2 pipeline completed successfully!
📊 Final metrics:
   ✅ Status: SUCCESS
   📖 Total rows processed: 10 → 10
   🔄 Data throughput: 500.0 MB

📋 Recent runs for ntk_Sil2Gld_DimPermission V2:
+-----+------+---------+-------+---------------+--------+-----------+
|LogID|Status|StartTime|EndTime|DurationSeconds|RowsRead|RowsWritten|
+-----+------+---------+-------+---------------+--------+-----------+
+-----+------+---------+-------+---------------+--------+-----------+


🧹 Cleaning up variables...
✨ ntk_Sil2Gld_DimPermission V2 logging completed!
